# Seguridad del servicio gRPC

Endurecemos el servidor gRPC del proyecto (`tp_mlops2.grpc_api.server`), que sirve el
mismo modelo que REST y GraphQL sobre `scoring.proto`.

**Riesgo:** el servidor real corre con `add_insecure_port` (sin TLS) y sin ningún
interceptor de autenticación — cualquiera que sepa la dirección y el puerto puede invocar
`Predict`/`PredictBatch`. Un canal sin cifrar también es vulnerable a *man-in-the-middle*
(un atacante en la misma red podría leer o modificar las predicciones en tránsito).

**Controles que agregamos:** un interceptor que exige un JWT válido en los metadatos de
cada llamada, límite de tamaño de mensaje (anti-DoS por payloads gigantes), y documentamos
el paso conceptual a TLS.

In [1]:
import datetime

import grpc
import jwt  # pyjwt
from concurrent import futures

from tp_mlops2.grpc_api import scoring_pb2, scoring_pb2_grpc
from tp_mlops2.grpc_api.server import ScoringServicer

servicer_real = ScoringServicer()  # carga el modelo real una sola vez, igual que en producción
print(f"Servicer real listo. model_version={servicer_real.model_version}")

2026/10/10 19:07:51 INFO mlflow.agent.hint: Load the `instrumenting-with-mlflow-tracing` skill at D:\ESTUDIOS\posgrado\Materias\CEIA (5) Operaciones de aprendizaje de maquina 2\tp_mlops2\.venv\Lib\site-packages\mlflow\assistant\skills\instrumenting-with-mlflow-tracing\SKILL.md before writing any tracing code; it ships with this MLflow install. Set MLFLOW_DISABLE_AGENT_HINT=1 to silence this.


Servicer real listo. model_version=random_forest_demanda@production


## 1. Estado actual: sin autenticación, canal sin cifrar

Levantamos el servidor real tal como corre hoy (`add_insecure_port`, sin interceptores)
en un puerto de prueba, y confirmamos que cualquier cliente puede llamar sin credenciales.

In [2]:
PUERTO_VULNERABLE = 50061


def levantar_servidor_vulnerable():
    server = grpc.server(futures.ThreadPoolExecutor(max_workers=4))
    scoring_pb2_grpc.add_ScoringServicer_to_server(servicer_real, server)
    server.add_insecure_port(f"[::]:{PUERTO_VULNERABLE}")
    server.start()
    return server


server_v = levantar_servidor_vulnerable()

channel_v = grpc.insecure_channel(f"localhost:{PUERTO_VULNERABLE}")
stub_v = scoring_pb2_grpc.ScoringStub(channel_v)

request = scoring_pb2.PredictRequest(
    timestamp="2018-06-15T14:00:00",
    temp_madrid=28.0, temp_barcelona=25.0, temp_valencia=27.0,
    temp_seville=32.0, temp_bilbao=20.0,
    load_24h_ago=30000.0, load_168h_ago=29500.0,
)

respuesta = stub_v.Predict(request)
print("sin credenciales de ningún tipo:", respuesta.predicted_load_mw, "MW (nadie lo frena)")

channel_v.close()
server_v.stop(0)

sin credenciales de ningún tipo: 30694.077109585807 MW (nadie lo frena)


<threading.Event at 0x1a95b5e8050: unset>

## 2. Interceptor de autenticación JWT + límite de tamaño de mensaje

El interceptor verifica un JWT real (no un token fijo) en los metadatos de la llamada,
y el canal del servidor limita el tamaño máximo de un mensaje entrante.

In [3]:
JWT_SECRET = "cambiar-en-produccion-con-32-bytes-o-mas"  # en prod: desde el entorno
MAX_MESSAGE_BYTES = 4 * 1024 * 1024  # 4 MB: suficiente para un PredictRequest, no para abuso


def crear_token(usuario: str, rol: str, minutos: int = 15) -> str:
    payload = {
        "sub": usuario,
        "rol": rol,
        "exp": datetime.datetime.now(datetime.UTC) + datetime.timedelta(minutes=minutos),
    }
    return jwt.encode(payload, JWT_SECRET, algorithm="HS256")


class AuthInterceptor(grpc.ServerInterceptor):
    """Verifica un JWT en el metadata 'authorization' antes de dejar pasar la llamada."""

    def __init__(self):
        self._denegado = grpc.unary_unary_rpc_method_handler(
            lambda req, ctx: ctx.abort(grpc.StatusCode.UNAUTHENTICATED, "token inválido o ausente")
        )

    def intercept_service(self, continuation, handler_call_details):
        metadata = dict(handler_call_details.invocation_metadata or [])
        auth_header = metadata.get("authorization", "")

        if not auth_header.startswith("Bearer "):
            return self._denegado

        token = auth_header.removeprefix("Bearer ")
        try:
            jwt.decode(token, JWT_SECRET, algorithms=["HS256"])
        except jwt.PyJWTError:
            return self._denegado

        return continuation(handler_call_details)


PUERTO_SEGURO = 50062


def levantar_servidor_seguro():
    server = grpc.server(
        futures.ThreadPoolExecutor(max_workers=4),
        interceptors=[AuthInterceptor()],
        options=[
            ("grpc.max_receive_message_length", MAX_MESSAGE_BYTES),
            ("grpc.max_send_message_length", MAX_MESSAGE_BYTES),
        ],
    )
    scoring_pb2_grpc.add_ScoringServicer_to_server(servicer_real, server)
    server.add_insecure_port(f"[::]:{PUERTO_SEGURO}")  # TLS: ver celda de reflexión
    server.start()
    return server


server_s = levantar_servidor_seguro()
channel_s = grpc.insecure_channel(f"localhost:{PUERTO_SEGURO}")
stub_s = scoring_pb2_grpc.ScoringStub(channel_s)

# sin token: rechazado
try:
    stub_s.Predict(request)
    print("sin token: no debería haber llegado acá")
except grpc.RpcError as e:
    print("sin token     :", e.code(), "(esperado UNAUTHENTICATED)")

# con token válido: pasa
token_ok = crear_token("alan", "analista")
md = (("authorization", f"Bearer {token_ok}"),)
respuesta_ok = stub_s.Predict(request, metadata=md)
print("con JWT válido:", respuesta_ok.predicted_load_mw, "MW")

# con token vencido: rechazado
token_vencido = jwt.encode(
    {"sub": "alan", "rol": "analista",
     "exp": datetime.datetime.now(datetime.UTC) - datetime.timedelta(minutes=1)},
    JWT_SECRET, algorithm="HS256",
)
try:
    stub_s.Predict(request, metadata=(("authorization", f"Bearer {token_vencido}"),))
    print("token vencido: no debería haber llegado acá")
except grpc.RpcError as e:
    print("token vencido :", e.code(), "(esperado UNAUTHENTICATED)")

channel_s.close()
server_s.stop(0)

sin token     : StatusCode.UNAUTHENTICATED (esperado UNAUTHENTICATED)
con JWT válido: 30694.077109585814 MW
token vencido : StatusCode.UNAUTHENTICATED (esperado UNAUTHENTICATED)


<threading.Event at 0x1a95a37ce60: unset>

## 3. De `insecure_port` a TLS (conceptual)

El interceptor protege la *autenticación*, pero no el *canal*: sin TLS, un atacante en la
misma red todavía puede leer el contenido de los mensajes (incluyendo el JWT que viaja en
los metadatos). El cambio real en este proyecto sería:

```python
with open("server.key", "rb") as f:
    clave_privada = f.read()
with open("server.crt", "rb") as f:
    certificado = f.read()

credenciales = grpc.ssl_server_credentials([(clave_privada, certificado)])
server.add_secure_port("[::]:50051", credenciales)  # en vez de add_insecure_port
```

Y del lado del cliente, `grpc.secure_channel(..., grpc.ssl_channel_credentials())` en vez
de `grpc.insecure_channel`. No lo activamos en este notebook porque requiere certificados
reales (o self-signed para desarrollo) y queda fuera del alcance de una demostración en
memoria — se documenta el cambio exacto para dejarlo listo si el servicio se expusiera
fuera de la red interna de Docker Compose.

## 4. Reflexión

**¿Qué parte del ataque frena el interceptor y qué parte frena TLS?**
El interceptor frena el **acceso no autenticado**: nadie sin un JWT válido puede invocar
`Predict`/`PredictBatch`, sin importar si conoce la dirección y el puerto del servicio.
TLS frena un problema distinto, a nivel de **transporte**: sin él, aunque la autenticación
sea correcta, un atacante con acceso a la red (dentro del mismo Docker network, por
ejemplo, si se compromete otro contenedor) puede leer el JWT y los datos de la predicción
en tránsito, o modificarlos en vuelo (*man-in-the-middle*). Son controles complementarios,
no sustitutos.

**¿Para qué sirve el límite de tamaño de mensaje?**
Sin él, un cliente (malicioso o con un bug) podría mandar un `PredictRequest` con un campo
anormalmente grande, forzando al servidor a asignar memoria desproporcionada para
deserializarlo — un vector de denegación de servicio simple pero real contra cualquier
servicio que acepte mensajes de tamaño variable.

**¿Qué elemento de SAIF cubre este control?**
El interceptor de JWT es *extender los fundamentos de seguridad al sistema de IA*: gRPC
no tiene autenticación por defecto como sí la tiene, por ejemplo, un API gateway REST
típico, así que hay que construirla explícitamente en la capa de transporte del servicio
de modelo.